# Task D — прогноз габаритов товара

Мультимодальная регрессия веса и трёх размеров товара по карточке и изображению.


## Зафиксированный результат

**Leaderboard score: 0.764** — лучший из сохранённых локальных вариантов задачи D.

> Метрика перенесена из авторского экспериментального ноутбука. Тяжёлые логи обучения удалены, чтобы решение хорошо отображалось на GitHub.


## Запуск

Положите данные в каталог `data/`, установите зависимости из корневого `requirements.txt` и последовательно выполните ячейки. Артефакты и сабмит будут записаны в `outputs/`.


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from PIL import Image
import torch
import timm # Для SigLIP
import torchvision.transforms as transforms
from tqdm import tqdm
import gc
from catboost import CatBoostRegressor

# --- НАСТРОЙКИ ---
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")

TRAIN_DIR = DATA_DIR / "train_images"
TEST_DIR = DATA_DIR / "test_images"

# 1. Загрузка данных
print("Loading data...")
train = pd.read_parquet(DATA_DIR / "train.parquet")
test = pd.read_parquet(DATA_DIR / "test.parquet")

# 2. Препроцессинг
def preprocess_data(df):
    df = df.copy()
    df["item_condition"] = df["item_condition"].fillna("Б/у")
    df["title"] = df["title"].fillna("")
    df["description"] = df["description"].fillna("")
    
    df["order_date"] = pd.to_datetime(df["order_date"])
    df["year"] = df["order_date"].dt.year
    df['month'] = df['order_date'].dt.month
    df["day"] = df["order_date"].dt.day
    
    df = df.drop(columns=["order_date", "item_id"]) 
    return df

train = preprocess_data(train)
test = preprocess_data(test)

# 3. Статистика по группам
print("Calculating group statistics...")
group_stats = train.groupby("microcat_name")[['real_weight', "real_height", "real_length", "real_width"]].agg(['mean', "std"])
group_stats.columns = ["_".join(col).strip() for col in group_stats.columns.values]
group_stats = group_stats.reset_index()

train = train.merge(group_stats, on="microcat_name", how="left")
test = test.merge(group_stats, on="microcat_name", how="left")
test = test.fillna(0) 

del group_stats
gc.collect()

# 4. Подготовка X и y
X_train_orig = train.drop(columns=["real_weight", "real_height", "real_length", "real_width"])
y_cols = ["real_weight", "real_height", "real_length", "real_width"]
y = train[y_cols]

all_data = pd.concat([X_train_orig, test], axis=0).reset_index(drop=True)

# 5. TF-IDF + SVD (Вместо BERT)
print("Processing text with TF-IDF...")
cat_feat = all_data.select_dtypes(include="object").columns
cat_feat_text = [c for c in cat_feat if c != 'image_name']

all_data['text_tfidf'] = all_data[cat_feat_text].astype(str).agg(" ".join, axis=1)

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,3)) # Увеличил max_features, так как нет BERT
tf_matrix = tfidf.fit_transform(all_data['text_tfidf'])

svd_tfidf = TruncatedSVD(n_components=100, random_state=42) # Сжимаем до 100 признаков
svd_mat = svd_tfidf.fit_transform(tf_matrix)

df_svd = pd.DataFrame(svd_mat, columns=[f"tfidf_{i}" for i in range(100)], index=all_data.index)
all_data = pd.concat([all_data, df_svd], axis=1).drop(columns=['text_tfidf'])

del tf_matrix, svd_mat, df_svd
gc.collect()

# 6. IMAGE EMBEDDINGS (SigLIP)
print("Loading SigLIP model...")
img_model = timm.create_model('vit_base_patch16_siglip_224', pretrained=True, num_classes=0).to(DEVICE)
img_model.eval()

data_config = timm.data.resolve_model_data_config(img_model)
img_transform = timm.data.create_transform(**data_config, is_training=False)

def get_img_embeddings(img_ids, img_dir, batch_size=32):
    embeddings_list = []
    img_paths = [str(Path(img_dir) / img_name) for img_name in img_ids]
    
    for i in tqdm(range(0, len(img_paths), batch_size), desc=f"Processing {Path(img_dir).name}"):
        batch_paths = img_paths[i:i+batch_size]
        images = []
        for p in batch_paths:
            try:
                img = Image.open(p).convert("RGB")
                images.append(img_transform(img))
            except:
                images.append(torch.zeros(3, 224, 224))
        
        batch = torch.stack(images).to(DEVICE)
        with torch.no_grad():
            emb = img_model(batch).cpu().numpy()
        embeddings_list.append(emb)
        
    return np.vstack(embeddings_list)

print("Extracting Train Image Embeddings (SigLIP)...")
train_img_emb = get_img_embeddings(train['image_name'].tolist(), TRAIN_DIR)

print("Extracting Test Image Embeddings (SigLIP)...")
test_img_emb = get_img_embeddings(test['image_name'].tolist(), TEST_DIR)

# СЖАТИЕ ЭМБЕДДИНГОВ ИЗОБРАЖЕНИЙ
print("Compressing image embeddings...")
svd_img = TruncatedSVD(n_components=767, random_state=42)
full_img_emb = np.vstack([train_img_emb, test_img_emb])
img_compressed = svd_img.fit_transform(full_img_emb)

n_train = len(train)
df_img_train = pd.DataFrame(img_compressed[:n_train], columns=[f'img_{i}' for i in range(767)])
df_img_test = pd.DataFrame(img_compressed[n_train:], columns=[f'img_{i}' for i in range(767)])

# Надежная склейка с сбросом индексов
all_data_reset = all_data.reset_index(drop=True)
df_img_full = pd.concat([df_img_train, df_img_test], ignore_index=True)
all_data = pd.concat([all_data_reset, df_img_full], axis=1)

del train_img_emb, test_img_emb, full_img_emb, img_compressed, img_model, df_img_train, df_img_test, all_data_reset, df_img_full
gc.collect()

# 7. ФИНАЛЬНАЯ ПОДГОТОВКА
X_final = all_data.iloc[:len(train)].drop(columns=["image_name"])
X_test_final = all_data.iloc[len(train):].drop(columns=["image_name"])

cat_features_final = ["item_condition", "category_name", "subcategory_name", "microcat_name"]

X_tr, X_val, y_tr, y_val = train_test_split(X_final, y, test_size=0.1, random_state=42)
y_tr_log = np.log1p(y_tr)
y_val_log = np.log1p(y_val)

# 8. ОБУЧЕНИЕ CATBOOST
predictions = {}
for col in y_cols:
    print(f"Training model for: {col}")
    cb = CatBoostRegressor(
        iterations=1500, 
        learning_rate=0.03, 
        loss_function='MAE', 
        verbose=10,
        random_seed=42
    )
    cb.fit(
        X_tr, 
        y_tr_log[col], 
        eval_set=(X_val, y_val_log[col]), 
        cat_features=cat_features_final,
        use_best_model=True
    )
    predictions[col] = np.expm1(cb.predict(X_test_final))

# 9. САБМИТ
submission_s = pd.read_csv(DATA_DIR / "sample_submission.csv")
sub = pd.DataFrame({
    "item_id": submission_s["item_id"],
    "target_weight": predictions["real_weight"],
    "target_height": predictions["real_height"],
    "target_length": predictions["real_length"],
    "target_width": predictions["real_width"],
})

sub.to_csv(OUTPUT_DIR / "submission_siglip_hybrid.csv", index=False)
print("Done! Submission saved.")
